In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/fatiheogan/OttoLatin.git
%cd /content/OttoLatin
!pip install -q -U transformers accelerate peft datasets

fatal: destination path 'OttoLatin' already exists and is not an empty directory.
/content/OttoLatin


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train_small_backup.zip to train_small_backup (1).zip


In [ ]:
import os
import shutil
import zipfile

zip_path = "/content/OttoLatin/train_small_backup.zip"
extract_dir = "/content/qwen_data"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

source = "/content/qwen_data/train_small.jsonl"
target_dir = "/content/OttoLatin/data/processed"
os.makedirs(target_dir, exist_ok=True)

shutil.copy2(source, f"{target_dir}/train_small.jsonl")

print("Training data copied successfully.")
print("Size:", os.path.getsize(f"{target_dir}/train_small.jsonl"), "bytes")

Training data copied successfully.
Size: 63058054 bytes


In [ ]:
%cd /content/OttoLatin

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded:", MODEL_NAME)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


/content/OttoLatin


W1110 18:49:58.330000 4015 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 18:49:58.370000 4015 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
CUDA: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Attach LoRA adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load training data
dataset = load_dataset(
    "json",
    data_files="/content/OttoLatin/data/processed/train_small.jsonl",
)["train"]

SYSTEM_PROMPT = (
    "You transliterate Ottoman Turkish written in Arabic script "
    "into Modern Turkish Latin script. Output only the transliteration."
)

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["ottoman"]},
        {"role": "assistant", "content": example["turkish"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
    desc="Formatting chat examples",
)

def tokenize_example(batch):
    return tokenizer(
        batch["text"],
        max_length=512,
        truncation=True,
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_example,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing chat examples",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(tokenized_dataset)
print("Qwen LoRA data preparation completed.")

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Generating train split: 0 examples [00:00, ? examples/s]

Formatting chat examples:   0%|          | 0/300000 [00:00<?, ? examples/s]

Tokenizing chat examples:   0%|          | 0/300000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 300000
})
Qwen LoRA data preparation completed.


In [ ]:
!pip install -q -U "torchao>=0.16.0"

In [ ]:
from transformers import Trainer, TrainingArguments

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/OttoLatin/checkpoints/qwen_lora",
    max_steps=1000,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting Qwen LoRA training...")
trainer.train()
print("Qwen LoRA training completed.")

Starting Qwen LoRA training...


Step,Training Loss
50,4.010833
100,2.226047
150,2.020324
200,1.890454
250,1.787996
300,1.748185
350,1.697002
400,1.713817
450,1.685454
500,1.642646


Qwen LoRA training completed.


In [ ]:
import json
import os
import torch

model.config.use_cache = True
model.eval()

tokenizer.padding_side = "left"

PRED_DIR = "/content/OttoLatin/preds/qwen_lora"
os.makedirs(PRED_DIR, exist_ok=True)

SETS = ["test_2k", "riqa", "naskh", "ot2tr"]
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 512

SYSTEM_PROMPT = (
    "You transliterate Ottoman Turkish written in Arabic script "
    "into Modern Turkish Latin script. Output only the transliteration."
)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def build_prompt(ottoman):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ottoman},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

for set_name in SETS:
    input_path = f"/content/OttoLatin/data/eval/{set_name}.jsonl"
    output_path = f"{PRED_DIR}/{set_name}.jsonl"

    rows = load_jsonl(input_path)
    predictions = []

    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        prompts = [build_prompt(row["ottoman"]) for row in batch]

        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=256,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_length = encoded["input_ids"].shape[1]
        generated_only = generated[:, prompt_length:]

        decoded = tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
        )

        for row, prediction in zip(batch, decoded):
            predictions.append({
                "id": str(row["id"]),
                "prediction": prediction.strip().split("\n")[0].strip(),
            })

        if start % (BATCH_SIZE * 50) == 0:
            print(
                f"{set_name}: "
                f"{min(start + BATCH_SIZE, len(rows))}/{len(rows)}"
            )

    with open(output_path, "w", encoding="utf-8") as f:
        for prediction in predictions:
            f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

    print(f"Saved {len(predictions)} predictions: {output_path}")

print("All Qwen LoRA predictions completed.")

test_2k: 4/2000
test_2k: 204/2000
test_2k: 404/2000
test_2k: 604/2000
test_2k: 804/2000
test_2k: 1004/2000
test_2k: 1204/2000
test_2k: 1404/2000
test_2k: 1604/2000
test_2k: 1804/2000
Saved 2000 predictions: /content/OttoLatin/preds/qwen_lora/test_2k.jsonl
riqa: 4/1575
riqa: 204/1575
riqa: 404/1575
riqa: 604/1575
riqa: 804/1575
riqa: 1004/1575
riqa: 1204/1575
riqa: 1404/1575
Saved 1575 predictions: /content/OttoLatin/preds/qwen_lora/riqa.jsonl
naskh: 4/376
naskh: 204/376
Saved 376 predictions: /content/OttoLatin/preds/qwen_lora/naskh.jsonl
ot2tr: 4/2358
ot2tr: 204/2358
ot2tr: 404/2358
ot2tr: 604/2358
ot2tr: 804/2358
ot2tr: 1004/2358
ot2tr: 1204/2358
ot2tr: 1404/2358
ot2tr: 1604/2358
ot2tr: 1804/2358
ot2tr: 2004/2358
ot2tr: 2204/2358
Saved 2358 predictions: /content/OttoLatin/preds/qwen_lora/ot2tr.jsonl
All Qwen LoRA predictions completed.


In [ ]:
%cd /content/OttoLatin

import json
import os
import zipfile

sets = {
    "test_2k": 2000,
    "riqa": 1575,
    "naskh": 376,
    "ot2tr": 2358,
}

pred_dir = "preds/qwen_lora"

for set_name, expected_count in sets.items():
    path = f"{pred_dir}/{set_name}.jsonl"

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    assert len(rows) == expected_count
    assert all("id" in row and "prediction" in row for row in rows)
    assert len({row["id"] for row in rows}) == expected_count

    print(f"{set_name}: {len(rows)} rows - OK")

zip_path = "preds_qwen_lora.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for set_name in sets:
        archive.write(
            f"{pred_dir}/{set_name}.jsonl",
            arcname=f"qwen_lora/{set_name}.jsonl",
        )

print(f"Created: {zip_path}")

/content/OttoLatin
test_2k: 2000 rows - OK
riqa: 1575 rows - OK
naskh: 376 rows - OK
ot2tr: 2358 rows - OK
Created: preds_qwen_lora.zip


In [ ]:
from google.colab import files

files.download("/content/OttoLatin/preds_qwen_lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>